# Final Project - ST 554
Author: Max Campbell

## Part 1 - Fitting a model using MLlib

In this assignment, we will demonstrate the capabilities of building models via pySpark using machine learning tools and data streaming! In particular, we want to build a pipeline that we can use to fit data to an elastic net model (chosen because we can cross-validate to find the optimal tuning parameters, which in turn will allow the model to keep itself relatively stable at high levels of complexity), and use that pipeline to predict new data against the model quickly and effectively. Let's begin by reading in the base dataset that we will use to fit the model. Our dataset of choice is power readings from Tetouan, Morocco as it relates to various environmental factors such as temperature, humidity, and time of day. The variable of interest is `Power_Zone_3`, representing power readings from a subsection of the city. In this context, we imagine that we are anticipating the tools used to measure `Power_Zone_3` are going offline soon, and we want a way to predict the power output while the tools are offline. Let's go ahead and get started!

In [1]:
#Load in necessary modules
import pandas as pd
from pyspark.sql import SparkSession

#Read in data as a pandas DataFrame
power = pd.read_csv("power_ml_data.csv")

#Initialize Spark session
spark = SparkSession.builder.master('local[*]').appName('FP') \
    .config("spark.sql.ansi.enabled", "false").getOrCreate()

#Read pandas DF into Spark
power = spark.createDataFrame(power)

power.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 15:32:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/27 15:32:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/27 15:32:57 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/27 15:32:57 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/04/27 15:32:57 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

Now that we've got our data in Spark, it's time to start setting up the pipeline. The transformations that we will be performing to this base dataset will also be performed on any future data that we read in, which is why using Spark/MLlib is a good tool for this job. Let's start with the `Hour` variable. We want to understand whether a power reading was taken (roughly) at night or day, so we will start with binarizing this variable to create an indicator of night-time readings vs day-time readings. Note that we may also have to convert `Hour` to a `DoubleType` first to accomplish this.

In [2]:
#Check data types for the dataframe
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [3]:
#Hour is a long, but we want double type, so we will convert it and then binarize it
#Import necessary modules
from pyspark.ml.feature import SQLTransformer, Binarizer
from pyspark.ml import Pipeline

#Convert Hour to Double, rename Power_Zone_3 to label
doubleTypeConverter = SQLTransformer(statement = "SELECT *, CAST(Hour AS DOUBLE) AS Hour_d, Power_Zone_3 AS label FROM __THIS__")

#Binarize Hour by whether the value is less than 6.5 or not
hourBinarizer = Binarizer(threshold = 6.5, inputCol = "Hour_d", outputCol = "isDaytime")

Next, we will need to one-hot encode the Month column so that the model can read the categorical data in a format that it can process efficiently.

In [4]:
#Import necessary modules
from pyspark.ml.feature import OneHotEncoder

#OHE Month
oneHotEncoder = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_ohe"])

After that, we will do a principal components analysis (PCA) on the environmental factors `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`. This will involve doing a PCA fit on these variables, which we will then use as the transformer that fits into our overall pipeline.

In [5]:
#Import necessary modules
from pyspark.ml.feature import VectorAssembler, PCA

#Assemble PCA features into a model
pcaVecAssembler = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol = "pcaFeatures")

#Fit the PCA model
pca = PCA(k = 2, inputCol = "pcaFeatures", outputCol = "pcaOutputs")

pipeline = Pipeline(stages = [doubleTypeConverter, hourBinarizer, oneHotEncoder, pcaVecAssembler, pca])
testing = pipeline.fit(power)

Now we can set up for the Elastic Net model, by assembling our desired features and defining the response variable.

In [6]:
#Define features
vecAssembler = VectorAssembler(inputCols = ["pcaFeatures", "isDaytime", "Power_Zone_1", "Power_Zone_2", "Month_ohe"], outputCol = "features")

We are ready to cross-validate the model now!

In [7]:
#Import necessary modules
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

#Define model
lr = LinearRegression()

#Define grid of parameters to cross-validate
grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()
    
#Define evaluator
evaluator = RegressionEvaluator(
    labelCol="label", 
    predictionCol="prediction", 
    metricName="rmse"
)

#Define pipeline
pipeline = Pipeline(stages = [doubleTypeConverter, hourBinarizer, oneHotEncoder, pcaVecAssembler, pca, vecAssembler, lr])

#Define CV
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = grid,
                          evaluator = evaluator,
                          parallelism = 128,
                          numFolds = 5)

#Fit power data to CV
fitted_crossval = crossval.fit(power)

26/04/27 15:33:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/27 15:33:30 WARN Instrumentation: [699b7cc9] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:30 WARN Instrumentation: [6fca5550] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:30 WARN Instrumentation: [72cd797d] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:30 WARN Instrumentation: [4f2b642e] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:30 WARN Instrumentation: [fd98b319] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:31 WARN Instrumentation: [1fea7d86] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 15:33:32 WARN Instrumentation: [72ba9327] regP

In [13]:
#Obtain the best model and show the best parameter values
best_model = fitted_crossval.bestModel
best_lr_model = best_model.stages[-1] #Gets the model fit at the last stage of the pipeline

print("Regression Parameter: ", best_lr_model.getRegParam())
print("Elastic Net Parameter: ", best_lr_model.getElasticNetParam())

#Get CV error and print that as well
cv_err = sum(fitted_crossval.avgMetrics) / len(fitted_crossval.avgMetrics)
print("Average CV error: ", cv_err)

Regression Parameter:  0.5
Elastic Net Parameter:  0.05
Average CV error:  2124.963937978689


We see that our best performing model returned a regression parameter of 0.5 and an elastic net parameter of 0.05, so these are the values we will use in our model fit for predicting future data! Let's go ahead and compare this model to the whole training set now so we have an idea of what our Root Mean Squared Error (RMSE) is.

In [14]:
#Import necessary modules
import numpy as np

#Predict on the fitted model
prediction_df = fitted_crossval.transform(power)

#Calculate RMSE
rmse = evaluator.evaluate(prediction_df)
print("RMSE: ", rmse)

RMSE:  2124.154490323416


We are also interested in the residuals for these predictions. This will help us understand how close our predictions are on a case-by-case basis.

In [15]:
#Create a residual column
trimmed_df = prediction_df.withColumn("residual", prediction_df.label - prediction_df.prediction)[["label", "prediction", "residual"]]
trimmed_df.show()

+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386|20203.987122100694| 36.97673789930559|
|20131.08434| 18012.08308052219|2119.0012594778127|
|19668.43373|17558.846315787905|2109.5874142120956|
|18899.27711|16939.637697748807| 1959.639412251192|
|18442.40964|16338.102891094724|2104.3067489052773|
|18130.12048|15859.139619515281|2270.9808604847203|
|17945.06024|15419.933509440893|2525.1267305591064|
|17459.27711|15041.889305957233| 2417.387804042766|
|17025.54217|14624.397962962601|2401.1442070373996|
|16794.21687|14285.106165766618|2509.1107042333824|
|16638.07229|14010.339151506538|2627.7331384934623|
|16395.18072|13779.185290140278| 2615.995429859722|
|16117.59036|13400.660687816198| 2716.929672183802|
| 15822.6506|12945.452316288818|2877.1982837111827|
|15672.28916|12776.870094494367|2895.4190655056336|
|15597.10843|12624.237465341179|2972.8709646588213|
|15510.36145

Now we have everything we need to begin evaluating new data!

## Part 2 - Streaming new data into a model

To begin, let's set up a stream reader that watches the `streamdata` folder in this project's directory for new CSV files.

In [16]:
#Set up a stream reader
stream = spark \
    .readStream \
    .schema(power.schema) \
    .option("header", "true") \
    .csv("streamdata")

Next, we want to do two separate things with the data we read in. The first is we want to create predictions and residuals for each new observation (and include `label`, representing `Power_Zone_3` as it did in the previous section). We can use the fitted model from the previous part as a transformer to accomplish this. The second thing we want to do is modify the `Power_Zone_3` column to read as `label`, so that we can demonstrate how performing a join on a stream works!

In [23]:
#Fit the new data and obtain the label, prediction, and residual reading

power_predict = fitted_crossval.transform(stream)[["label", "prediction"]]
power_predict = power_predict.withColumn("residual", power_predict.label - power_predict.prediction)
power_label = stream.withColumnRenamed("Power_Zone_3", "label")

#Perform an inner join on power_predict and power_label
power_joined = power_predict.join(power_label, "label", "inner")

Assuming everything went to plan, we should be good to start writing our stream to the console! Let's get it started and see.

In [24]:
#Write stream to console using append mode
query = power_joined.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/27 15:49:38 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-fcc5e774-26f5-480e-8f67-b085d64f83d3. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 15:49:38 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [ ]:
#Manually stop query if necessary
query.stop()